# Embedding predictions of ogbn network classes

In this file I will focus on making class predictions through the use of node_feature embedding space.

The embedding vectors have a dimension of 128, and there are about 170k nodes in total.

## Imports and dataset download

In [73]:
!pip install ogb --quiet

In [74]:
from ogb.nodeproppred import NodePropPredDataset
import numpy as np
import torch

In [76]:
import torch

# Save original torch.load function
_torch_load_func = torch.load

# Define a wrapper that forces weights_only=False
def custom_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _torch_load_func(*args, **kwargs)

# Monkey-patch torch.load to use our custom wrapper
torch.load = custom_torch_load

# Initialize the dataset (NodePropPredDataset is imported in a previous cell)
dataset = NodePropPredDataset(name='ogbn-arxiv')
graph, y = dataset[0]

In [77]:
y = y.squeeze()
y

array([ 4,  5, 28, ..., 10,  4,  1])

In [78]:
graph["node_feat"]

array([[-0.057943, -0.05253 , -0.072603, ...,  0.173364, -0.172796,
        -0.140059],
       [-0.1245  , -0.070665, -0.325202, ...,  0.068524, -0.372111,
        -0.301036],
       [-0.080242, -0.023328, -0.183787, ...,  0.109919,  0.117589,
        -0.139883],
       ...,
       [-0.22053 , -0.036568, -0.402199, ...,  0.11336 , -0.161393,
        -0.145171],
       [-0.138236,  0.040885, -0.251811, ..., -0.08929 , -0.041253,
        -0.376132],
       [-0.029875,  0.268417, -0.161124, ...,  0.120807,  0.077647,
        -0.091018]], dtype=float32)

In [79]:
split_idx = dataset.get_idx_split()

train_idx = split_idx["train"]

In [80]:
test_idx = split_idx["test"]

In [81]:
test_idx[0]

np.int64(346)

get centroids of all classes

That means go through each class and its nodes, get the node_features and compute mean

In [82]:
cls_indexes = []
for cls in range(40):
  cls_indexes.append(np.where(y == cls)[0])

In [83]:
cls_indexes[2]

array([    43,     79,    110, ..., 169111, 169302, 169315])

## Compute centroids on the entire train dataset

Get all features from `cls_indexes[x]` and compute mean

In [ ]:
centroids = []
for cls, indexes in enumerate(cls_indexes):
  centroids.append(graph["node_feat"][indexes].mean(axis=0))


In [ ]:
centroids[0]

array([-0.08344597, -0.01832454, -0.21693629, -0.10920637,  0.02322552,
       -0.04646448, -0.39856356, -0.13793719,  0.00313103,  0.14164943,
       -0.06879787, -0.47172636,  0.15345055,  0.03252691, -0.20918609,
        0.04877224,  0.10530245,  0.43524104,  0.00213274, -0.19870794,
       -0.26638135,  0.21643046, -0.28756952,  0.06531173, -0.14859931,
        0.3776644 ,  0.07981879, -0.39428833,  0.0315932 ,  0.00891967,
       -0.09397177,  0.09271468, -0.08307739,  0.07854996, -0.00924365,
       -0.28972843, -0.10235976,  0.07824453,  0.11052133,  0.21274708,
        0.2284355 , -0.14059733,  0.15880881, -0.09098744,  0.11548962,
        0.0987733 ,  0.25682434, -0.04055779,  0.3263117 ,  0.12785828,
        0.29353216,  0.13361388, -0.09911864, -0.10702757, -0.12154882,
        0.04354135, -0.08090022, -0.0408359 , -0.00113526, -0.05026219,
        0.4231157 ,  0.04613413,  0.10808315, -0.02365121, -0.01431668,
        0.33283108, -0.23046419, -0.02567738,  0.3877431 , -0.38

Technically we could change the centroid computation to a weighted one, defined by how many edges a node has compared to the others.

The weighted centroid $\mathbf{c}$ of a set of embedding vectors $\{\mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_n\}$ corresponding to nodes with degrees $\{d_1, d_2, \dots, d_n\}$ is defined as:

$$\mathbf{c} = \frac{\sum_{i=1}^{n} d_i \mathbf{v}_i}{\sum_{i=1}^{n} d_i}$$

In [ ]:
for node in test_idx:
  node_features = graph["node_feat"][node]

  dot_products = np.dot(centroids, node_features)
  node_features_norm = np.linalg.norm(node_features)

  centroid_norms = np.linalg.norm(centroids, axis=1)
  similarities = dot_products / (node_features_norm * centroid_norms)
  print(similarities.argmax())
  break

10


In [ ]:
# Evaluate accuracy on the test set using the centroids
preds = []

# Converting centroids to a numpy array for vectorized computation
centroids_arr = np.array(centroids)
centroid_norms = np.linalg.norm(centroids_arr, axis=1)

# We'll check a subset or the whole test set
for node in test_idx:
    node_feat = graph["node_feat"][node]

    # Vectorized cosine similarity
    dot = np.dot(centroids_arr, node_feat)
    node_norm = np.linalg.norm(node_feat)
    sims = dot / (node_norm * centroid_norms)

    preds.append(sims.argmax())

# Calculate accuracy
correct = (np.array(preds) == y[test_idx])
accuracy = correct.mean()
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.3338


In [ ]:
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

In [ ]:
y_true = y[test_idx]

nmi = normalized_mutual_info_score(y_true, preds)
ari = adjusted_rand_score(y_true, preds)

print(f"NMI: {nmi:.4f}")
print(f"ARI: {ari:.4f}")

NMI: 0.2550
ARI: 0.1480


Note that this was not done through few-shot! centroids were computed as a mean of all the train nodes for each class, without care for their degree or possible weights.

## Weighted centroid predictions

In [ ]:
# edge_index has shape [2, num_edges]. edge_index[0] contains source nodes.
num_nodes = graph['num_nodes']
edge_index = graph['edge_index']

# Compute out-degree for each node
degrees = np.zeros(num_nodes)
sources, counts = np.unique(edge_index[0], return_counts=True)
degrees[sources] = counts

# To avoid division by zero and give isolated nodes some weight, we can use (degree + 1)
weights = degrees + 1

# Compute Weighted Centroids
weighted_centroids = []
for cls, indexes in enumerate(cls_indexes):
    cls_feats = graph["node_feat"][indexes]
    cls_weights = weights[indexes].reshape(-1, 1)

    weighted_avg = np.sum(cls_feats * cls_weights, axis=0) / np.sum(cls_weights)
    weighted_centroids.append(weighted_avg)

weighted_centroids_arr = np.array(weighted_centroids)

In [ ]:
# Evaluate accuracy with Weighted Centroids
w_preds = []
w_centroid_norms = np.linalg.norm(weighted_centroids_arr, axis=1)

for node in test_idx:
    node_feat = graph["node_feat"][node]
    dot = np.dot(weighted_centroids_arr, node_feat)
    node_norm = np.linalg.norm(node_feat)
    sims = dot / (node_norm * w_centroid_norms)
    w_preds.append(sims.argmax())

w_accuracy = (np.array(w_preds) == y[test_idx]).mean()
print(f"Weighted Centroid Test Accuracy: {w_accuracy:.4f}")
print(f"Improvement: {w_accuracy - accuracy:.4f}")

Weighted Centroid Test Accuracy: 0.3047
Improvement: -0.0291


In [ ]:
y_true = y[test_idx]

nmi = normalized_mutual_info_score(y_true, w_preds)
ari = adjusted_rand_score(y_true, w_preds)

print(f"NMI: {nmi:.4f}")
print(f"ARI: {ari:.4f}")

NMI: 0.2510
ARI: 0.1221


This suggests that in the dataset, a node's 'influence' doesn't necessarily make its features more representative of its class's center. In fact, highly cited papers or papers with many references might have more 'general' or interdisciplinary language, which could pull the centroid away from the specific feature clusters of that category.

## Few-shot centroid predictions

In [96]:
centroids = []
for cls, indexes in enumerate(cls_indexes):
  centroids.append(graph["node_feat"][np.random.choice(indexes, 16)].mean(axis=0))

In [92]:
for node in test_idx:
  node_features = graph["node_feat"][node]

  dot_products = np.dot(centroids, node_features)
  node_features_norm = np.linalg.norm(node_features)

  centroid_norms = np.linalg.norm(centroids, axis=1)
  similarities = dot_products / (node_features_norm * centroid_norms)
  print(similarities.argmax())
  break

30


In [97]:
# Evaluate accuracy on the test set using the centroids
preds = []

# Converting centroids to a numpy array for vectorized computation
centroids_arr = np.array(centroids)
centroid_norms = np.linalg.norm(centroids_arr, axis=1)

# We'll check a subset or the whole test set
for node in test_idx:
    node_feat = graph["node_feat"][node]

    # Vectorized cosine similarity
    dot = np.dot(centroids_arr, node_feat)
    node_norm = np.linalg.norm(node_feat)
    sims = dot / (node_norm * centroid_norms)

    preds.append(sims.argmax())

# Calculate accuracy
correct = (np.array(preds) == y[test_idx])
accuracy = correct.mean()
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.2952


In [98]:
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

In [99]:
y_true = y[test_idx]

nmi = normalized_mutual_info_score(y_true, preds)
ari = adjusted_rand_score(y_true, preds)

print(f"NMI: {nmi:.4f}")
print(f"ARI: {ari:.4f}")

NMI: 0.2210
ARI: 0.1268


With using the few-shot method, our accuracy drops below 29%.

## Deep Learning experimentation

Below are some tests for using DL models for prediction of classes using the embedding space.

The idea is that NNs can learn way more complex shapes.

In [100]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. Prepare Data
X_train = torch.tensor(graph["node_feat"][train_idx], dtype=torch.float32)
y_train = torch.tensor(y[train_idx], dtype=torch.long)
X_test = torch.tensor(graph["node_feat"][test_idx], dtype=torch.float32)
y_test = torch.tensor(y[test_idx], dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=512, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=512, shuffle=False)

# 2. Define a Simple MLP
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(MLPClassifier, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.layers(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLPClassifier(input_dim=128, hidden_dim=256, num_classes=40).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# 3. Training Loop
model.train()
for epoch in range(10):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        out = model(batch_x)
        loss = criterion(out, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# 4. Evaluation
model.eval()
correct = 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        preds = model(batch_x).argmax(dim=1)
        correct += (preds == batch_y).sum().item()

dl_accuracy = correct / len(test_idx)
print(f"\nDeep Learning (MLP) Test Accuracy: {dl_accuracy:.4f}")

Epoch 1, Loss: 1.9472
Epoch 2, Loss: 1.6794
Epoch 3, Loss: 1.6381
Epoch 4, Loss: 1.6136
Epoch 5, Loss: 1.5960
Epoch 6, Loss: 1.5763
Epoch 7, Loss: 1.5661
Epoch 8, Loss: 1.5547
Epoch 9, Loss: 1.5489
Epoch 10, Loss: 1.5428

Deep Learning (MLP) Test Accuracy: 0.5415
